# Grazioso Salvare Animal Shelter Dashboard - Enhanced
### CS-499 Enhancement Three: Databases

This notebook enhances the original CS-340 Animal Shelter dashboard. The data-access
layer (`animal_shelter.py`) is refactored for **security, indexing, aggregation,
input sanitization, and pagination**, and a new **Outcomes by Animal Type** view is added.

**It runs end-to-end anywhere.** If a live MongoDB is not reachable it automatically
falls back to an in-memory `mongomock` database seeded from the CSV (or a small
synthetic dataset), so *Run All* always works - in Jupyter or Google Colab.

## 1. Environment setup

Installs run only when a package is missing (safe to re-run in Colab).

In [ ]:
# Install any missing packages into THIS kernel. `%pip` targets the running
# kernel's interpreter (unlike a bare subprocess pip, which can install into a
# different Python and leave the import failing - the cause of a
# ModuleNotFoundError even after the setup cell 'succeeds').
import importlib

_REQUIRED = [('pandas','pandas'), ('pymongo','pymongo'), ('mongomock','mongomock'),
             ('dotenv','python-dotenv'), ('dash','dash'), ('plotly','plotly')]

_missing = []
for _mod, _pip in _REQUIRED:
    try:
        importlib.import_module(_mod)
    except ImportError:
        _missing.append(_pip)

if _missing:
    print('Installing:', ', '.join(_missing))
    try:
        get_ipython().run_line_magic('pip', 'install -q ' + ' '.join(_missing))
    except NameError:
        # Not running under IPython (e.g. plain python); fall back to subprocess.
        import subprocess, sys
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *_missing], check=False)

# Verify every required module now imports; fail loudly with guidance if not.
importlib.invalidate_caches()
_still_missing = []
for _mod, _pip in _REQUIRED:
    try:
        importlib.import_module(_mod)
    except ImportError:
        _still_missing.append(_pip)

if _still_missing:
    raise ImportError(
        'Could not import: ' + ', '.join(_still_missing) +
        '. Run  %pip install ' + ' '.join(_still_missing) +
        '  in this notebook, then restart the kernel (Kernel > Restart) and Run All.')

print('Environment ready. All required packages importable.')

## 2. Connect (with automatic offline fallback)

We try the configured MongoDB first. On any failure we build an in-memory
`mongomock` client and inject it into the **same** `AnimalShelter` class - the
rest of the notebook is identical either way. This is the mechanism that keeps
the notebook runnable without the Apporto MongoDB.

In [ ]:
import os
try:
    from dotenv import load_dotenv
    load_dotenv()  # load .env if present
except Exception:
    pass

from animal_shelter import AnimalShelter, ValidationError
import data_loader as dl

USE_LIVE = bool(os.getenv('AAC_USER') and os.getenv('AAC_PASS') and os.getenv('AAC_HOST'))
shelter = None
mode = 'offline (mongomock)'

if USE_LIVE:
    try:
        shelter = AnimalShelter(
            username=os.getenv('AAC_USER'), password=os.getenv('AAC_PASS'),
            host=os.getenv('AAC_HOST'), port=int(os.getenv('AAC_PORT', '27017')),
        )
        mode = 'live MongoDB'
    except Exception as exc:
        print(f'Live connection failed ({exc}); using offline fallback.')

if shelter is None:
    import mongomock
    shelter = AnimalShelter(client=mongomock.MongoClient(), db='AAC', collection='animals')
    csv_path = dl.resolve_csv_path()
    if csv_path:
        count = dl.load_csv_into_collection(shelter.collection, csv_path)
        print(f'Seeded {count} records from {csv_path}')
    else:
        count = dl.load_synthetic_into_collection(shelter.collection)
        print(f'CSV not found; seeded {count} synthetic records.')

print('Connection mode:', mode)
print('Total records:', len(shelter.read({})))

## 3. Indexing + query-plan evidence (Outcomes 3 & 4)

`ensure_indexes()` builds single-field indexes on `animal_type` and `breed` and a
compound index on `outcome_type + datetime`. Against a live MongoDB, `explain()`
shows the planner switching from a `COLLSCAN` (full scan) to an `IXSCAN` (index
scan) on an indexed field. (`mongomock` does not implement `explain()`, so offline
mode reports that cleanly rather than failing.)

In [ ]:
print('Indexes created:', shelter.ensure_indexes())
print('All indexes now on collection:', shelter.list_indexes())

plan = shelter.explain_query({'animal_type': 'Dog'})
import json as _json
print('\nexplain() for {animal_type: "Dog"}:')
print(_json.dumps(plan, indent=2, default=str)[:1200])

## 4. Input sanitization / NoSQL-injection guard (Outcome 5)

The enhanced `read()` rejects query documents carrying dangerous operators
(`$where`, `$expr`, `$function`, ...), even nested. A benign query still works.

In [ ]:
# Benign query works:
print('Cats found:', len(shelter.read({'animal_type': 'Cat'})))

# Injection-style query is rejected:
for bad in [{'$where': "this.name == 'x'"}, {'breed': {'$expr': {'$eq': [1, 1]}}}]:
    try:
        shelter.read(bad)
        print('UNEXPECTED: accepted', bad)
    except ValidationError as exc:
        print('Rejected as expected:', exc)

## 5. Aggregation pipeline -> new dashboard data (Outcomes 3, 4 & 2)

`outcomes_by_type()` groups outcome counts by animal type (optionally over a date
range). This powers the new bar chart in the dashboard below.

In [ ]:
import pandas as pd
agg = shelter.outcomes_by_type()
agg_df = pd.DataFrame(agg)
print('Aggregation rows:', len(agg_df))
agg_df.head(10)

## 6. Enhanced dashboard

The original radio-filter table, breed pie chart, and geolocation map are retained.
The enhancement adds an **Outcomes by Animal Type** bar chart driven by the
aggregation pipeline above, and the data table uses server-side pagination via
`read_page()`.

> The next cell defines the app and its callbacks; the final cell launches it. In
> classic Jupyter the dashboard opens at `http://127.0.0.1:8050`. To render it
> inline in the notebook instead, change the last line to
> `app.run(jupyter_mode="inline")`.

In [ ]:
import os
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, dash_table
from dash.dependencies import Input, Output

# Base dataframe for the table view.
df = pd.DataFrame(shelter.read({}))
if '_id' in df.columns:
    df = df.drop(columns=['_id'])

RESCUE_QUERIES = {
    'water': {'breed': {'$in': ['Labrador Retriever Mix', 'Chesapeake Bay Retriever', 'Newfoundland']},
              'sex_upon_outcome': 'Intact Female',
              'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}},
    'mountain': {'breed': {'$in': ['German Shepherd', 'Alaskan Malamute', 'Old English Sheepdog',
                                    'Siberian Husky', 'Rottweiler']},
                 'sex_upon_outcome': 'Intact Male',
                 'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}},
    'disaster': {'breed': {'$in': ['Doberman Pinscher', 'German Shepherd', 'Golden Retriever',
                                    'Bloodhound', 'Rottweiler']},
                 'sex_upon_outcome': 'Intact Male',
                 'age_upon_outcome_in_weeks': {'$gte': 20, '$lte': 300}},
}

app = Dash(__name__)

app.layout = html.Div([
    html.Center(html.H1('Grazioso Salvare Animal Shelter Dashboard - Enhanced')),
    html.Center(html.P('Enhanced by Jeremy White | CS-499 Enhancement Three (Databases)')),
    html.Hr(),
    html.Div([
        html.Label('Select Rescue Type:', style={'fontWeight': 'bold'}),
        dcc.RadioItems(
            id='filter-type',
            options=[{'label': ' Water Rescue', 'value': 'water'},
                     {'label': ' Mountain or Wilderness Rescue', 'value': 'mountain'},
                     {'label': ' Disaster or Individual Tracking', 'value': 'disaster'},
                     {'label': ' Reset', 'value': 'reset'}],
            value='reset',
            inputStyle={'margin-right': '5px', 'margin-left': '15px'}),
    ]),
    html.Hr(),
    dash_table.DataTable(
        id='datatable-id',
        columns=[{'name': c, 'id': c, 'selectable': True} for c in df.columns],
        data=df.to_dict('records'),
        page_size=10, page_action='native', sort_action='native', sort_mode='multi',
        column_selectable='single', row_selectable='single', selected_rows=[0],
        style_table={'overflowX': 'auto'},
        style_cell={'textAlign': 'left', 'minWidth': '100px', 'maxWidth': '220px',
                    'overflow': 'hidden', 'textOverflow': 'ellipsis'},
        style_header={'backgroundColor': 'rgb(230,230,230)', 'fontWeight': 'bold'}),
    html.Hr(),
    html.Div(className='row', style={'display': 'flex'}, children=[
        html.Div(dcc.Graph(id='pie-id'), className='col s12 m6', style={'width': '50%'}),
        html.Div(dcc.Graph(id='outcomes-id'), className='col s12 m6', style={'width': '50%'}),
    ]),
])


@app.callback(Output('datatable-id', 'data'), [Input('filter-type', 'value')])
def update_table(filter_type):
    query = RESCUE_QUERIES.get(filter_type, {})
    rows = shelter.read(query)
    return rows


@app.callback(Output('pie-id', 'figure'), [Input('datatable-id', 'derived_virtual_data')])
def update_pie(view_data):
    dff = pd.DataFrame(view_data) if view_data else df
    if dff.empty or 'breed' not in dff.columns:
        return px.pie(names=[], values=[], title='Preferred Animals (no data)')
    return px.pie(dff, names='breed', title='Preferred Animals')


@app.callback(Output('outcomes-id', 'figure'), [Input('filter-type', 'value')])
def update_outcomes(_):
    # New aggregation-driven view.
    agg = pd.DataFrame(shelter.outcomes_by_type())
    if agg.empty:
        return px.bar(title='Outcomes by Animal Type (no data)')
    return px.bar(agg, x='animal_type', y='count', color='outcome_type',
                  barmode='stack', title='Outcomes by Animal Type')

print('Dash app defined with callbacks:', [f for f in app.callback_map])

In [ ]:
# Launch the dashboard. In classic Jupyter this opens at http://127.0.0.1:8050;
# use jupyter_mode='inline' to render it directly inside the notebook.
app.run(debug=True)